# Lecture 2 — Bias-Variance Trade-Off & Model Selection

**Statistisches Lernen 2** — FH Kufstein Tirol  
Professor: Johannes Schwab, PhD

---

## O que estamos aprendendo nesta aula (Resumo em 5 linhas)

Esta aula aborda o **coração estratégico da modelagem preditiva**: o equilíbrio perfeito entre um modelo simplista demais (**High Bias**) e um modelo excessivamente complexo que decora o ruído (**High Variance**). Você aprenderá a decompor o erro matematicamente em três partes irredutíveis e, mais importante, dominará um **Framework prático de engenharia** para parar de adivinhar quais algoritmos usar. Substituiremos o "tentativa e erro" por regras de escolha baseadas no volume de dados ($N$ vs $p$), tipo de estrutura dos dados e definição de modelos de referência (**Baselines**). O objetivo final é agir como um cientista de dados sênior: eficiente, sistemático e guiado por evidências.

---

## Roteiro do Notebook

1. **The Core Dilemma** — Data Fit vs. Generalization  
2. **Mathematical Error Decomposition** — Dedução do $\text{Bias}^2 + \text{Variance} + \text{Irreducible Error}$  
3. **Visualizando o Trade-Off** — A curva clássica de complexidade vs. erro  
4. **Métodos para Balancear Bias e Variance** — Cross-Validation, Regularização, Feature Selection, Ensembles  
5. **Model Selection Rules of Thumb** — Heurísticas por tamanho de dados ($N$ e $p$)  
6. **Data Characteristics & Constraints** — Data Type, Noise Level, Interpretability  
7. **Structured 4-Phase Model Selection Workflow** — O framework corporativo completo  
8. **Metacognição: A Mentalidade do Cientista Preguiçoso** — Baselines antes de complexidade

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, KFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.datasets import make_regression
from sklearn.metrics import mean_squared_error

np.random.seed(42)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
print("Bibliotecas carregadas com sucesso.")

---

## 1. The Core Dilemma: Data Fit vs. Generalization

### Conceito Teórico

Todo modelo preditivo vive sob uma tensão fundamental entre dois objetivos conflitantes:

- **Data Fit (Ajuste aos dados de treino):** O modelo aprende bem os padrões — e também o ruído — dos dados que ele viu. Resultado: erros muito baixos no treino.
- **Generalization (Generalização):** O modelo consegue fazer previsões precisas para dados **nunca vistos antes**, que ele não usou para aprender.

> **É geralmente impossível maximizar ambos ao mesmo tempo.** Esta tensão irresolvível tem um nome: **Bias-Variance Trade-Off**.

**Analogia do Estudante:**
- O aluno que **decora as respostas** da lista de exercícios (overfitting) vai bem na lista, mas vai mal na prova com questões novas — **High Variance**.
- O aluno que **ignora os detalhes** e só usa uma regra geral muito simples vai errar tanto na lista quanto na prova — **High Bias**.
- O aluno ideal entende os conceitos e generaliza — **equilíbrio entre Bias e Variance**.

### Matemática

Formalmente, temos os dados $\{(x_j, y_j)\}_{j=1}^{K}$ gerados por:

$$y_j = f(x_j) + \epsilon_j, \quad \epsilon_j \sim \mathcal{N}(0, \sigma^2)$$

Nosso modelo treinado $\hat{f}$ deve minimizar o erro em **dados novos** (não no treino). O conflito:

| Cenário | Erro Treino | Erro Teste | Diagnóstico |
|---------|-------------|------------|-------------|
| Modelo muito simples | Alto | Alto | **Underfitting / High Bias** |
| Modelo ideal | Médio | Médio | **Boa generalização** |
| Modelo muito complexo | Baixo | Alto | **Overfitting / High Variance** |

In [ ]:
# Simulando o dilema central com dados não-lineares
f_true = lambda x: np.sin(1.5 * np.pi * x)

N_treino = 20
x_treino = np.sort(np.random.uniform(0, 2, N_treino))
y_treino = f_true(x_treino) + np.random.normal(0, 0.3, N_treino)

x_plot = np.linspace(0, 2, 400)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

configs = [
    (1,  'tab:red',   'Underfitting\n(High Bias)'),
    (5,  'tab:green', 'Bom equilíbrio\n(Low Bias, Low Variance)'),
    (18, 'tab:blue',  'Overfitting\n(High Variance)'),
]

for ax, (grau, cor, titulo) in zip(axes, configs):
    modelo = make_pipeline(PolynomialFeatures(grau), LinearRegression())
    modelo.fit(x_treino.reshape(-1, 1), y_treino)
    y_pred_treino = modelo.predict(x_treino.reshape(-1, 1))
    y_pred_plot  = modelo.predict(x_plot.reshape(-1, 1))

    mse_treino = mean_squared_error(y_treino, y_pred_treino)

    ax.scatter(x_treino, y_treino, s=50, color='gray', zorder=5, label='Treino')
    ax.plot(x_plot, f_true(x_plot), 'k--', alpha=0.4, lw=1.5, label='$f$ verdadeira')
    ax.plot(x_plot, y_pred_plot, '-', color=cor, lw=2.5, label=f'Polinômio grau {grau}')
    ax.set_title(f'{titulo}\nMSE treino = {mse_treino:.3f}')
    ax.set_ylim(-2.5, 2.5)
    ax.set_xlabel('x')
    ax.legend(fontsize=9)

plt.suptitle('The Core Dilemma: Data Fit vs. Generalization', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 2. Mathematical Error Decomposition

### Conceito Teórico

Por que o erro de teste é sempre maior que o erro de treino? Porque o erro total de um modelo é composto por **três partes distintas**, com causas e remédios diferentes. Compreender cada uma é fundamental para saber onde agir.

**Analogia do Arqueiro:**
- **Bias²:** O arqueiro mira sistematicamente à esquerda do alvo. Não importa quantas flechas ele dispare — todas erram no mesmo sentido. É um erro **estrutural do modelo**.
- **Variance:** O arqueiro mira bem, mas sua mão treme. As flechas caem em lugares aleatórios ao redor do alvo. É um erro de **sensibilidade aos dados de treino**.
- **Irreducible Error:** O vento sopra de forma imprevisível. Nada pode ser feito. É o **ruído intrínseco dos dados** ($\sigma^2$).

### Matemática — Dedução Completa

Queremos analisar o **Mean Squared Error (MSE)** do nosso modelo estimado $\hat{f}$ em relação ao verdadeiro $f$:

**Passo 1:** Expandindo $y = f(x) + \epsilon$:

$$\text{MSE}(x) = \mathbb{E}\left[(y - \hat{f}(x))^2\right] = \mathbb{E}\left[(f(x) + \epsilon - \hat{f}(x))^2\right]$$

**Passo 2:** Expandindo o quadrado:

$$= \mathbb{E}\left[(f(x) - \hat{f}(x))^2\right] + 2\mathbb{E}\left[(f(x) - \hat{f}(x))\,\epsilon\right] + \mathbb{E}\left[\epsilon^2\right]$$

**Passo 3:** O **termo cruzado desaparece** porque $\mathbb{E}[\epsilon] = 0$ e $\epsilon$ é independente de $x$:

$$2\mathbb{E}\left[(f(x) - \hat{f}(x))\,\epsilon\right] = 2(f(x) - \hat{f}(x))\cdot\underbrace{\mathbb{E}[\epsilon]}_{=\,0} = 0$$

O último termo é o **Irreducible Error**: $\mathbb{E}[\epsilon^2] = \sigma^2$. Logo:

$$\text{MSE}(x) = \mathbb{E}\left[(f(x) - \hat{f}(x))^2\right] + \sigma^2$$

**Passo 4:** Decompondo o primeiro termo adicionando e subtraindo $\mathbb{E}[\hat{f}(x)]$:

$$\mathbb{E}\left[(f(x) - \hat{f}(x))^2\right] = \mathbb{E}\left[(f(x) - \mathbb{E}[\hat{f}(x)] + \mathbb{E}[\hat{f}(x)] - \hat{f}(x))^2\right]$$

$$= \underbrace{\left(f(x) - \mathbb{E}[\hat{f}(x)]\right)^2}_{\text{Bias}^2} + \underbrace{\mathbb{E}\left[(\mathbb{E}[\hat{f}(x)] - \hat{f}(x))^2\right]}_{\text{Variance}} + \underbrace{2(f(x) - \mathbb{E}[\hat{f}(x)])\cdot\mathbb{E}[\mathbb{E}[\hat{f}(x)] - \hat{f}(x)]}_{= 0}$$

O último termo cruzado é zero pois $\mathbb{E}[\mathbb{E}[\hat{f}(x)] - \hat{f}(x)] = 0$ por definição de valor esperado.

**Resultado Final:**

$$\boxed{\text{Total Error} = \underbrace{\left(f(x) - \mathbb{E}[\hat{f}(x)]\right)^2}_{\text{Bias}^2} + \underbrace{\mathbb{E}\left[(\hat{f}(x) - \mathbb{E}[\hat{f}(x)])^2\right]}_{\text{Variance}} + \underbrace{\sigma^2}_{\text{Irreducible Error}}}$$

### Significado Estatístico de Cada Componente

| Componente | Fórmula | Significado | Causa | Remédio |
|------------|---------|-------------|-------|--------|
| **Bias²** | $(f - \mathbb{E}[\hat{f}])^2$ | Distância entre a previsão média e a verdade | Modelo muito simples (suposições erradas) | Aumentar complexidade, melhores features |
| **Variance** | $\mathbb{E}[(\hat{f} - \mathbb{E}[\hat{f}])^2]$ | Sensibilidade às mudanças nos dados de treino | Modelo muito complexo (decora o ruído) | Regularização, mais dados, Ensembles |
| **Irreducible Error** | $\sigma^2$ | Ruído intrínseco dos dados | Natureza dos dados | **Nada pode ser feito** |

> **Insight crucial:** o Irreducible Error é o teto de desempenho absoluto. Todo esforço de modelagem opera entre esse teto e a soma do Bias² + Variance.

In [ ]:
# Simulação empírica da decomposição Bias² + Variance + Irreducible Error
# Estratégia: treinar o mesmo modelo em muitos datasets diferentes amostrados da mesma distribuição

def simular_decomposicao(grau_poly, n_simulacoes=200, N=25, sigma_ruido=0.4):
    """Estima Bias², Variance e Irreducible Error empiricamente."""
    f_true = lambda x: np.sin(1.5 * np.pi * x)
    x_teste = np.linspace(0, 2, 50)
    f_verdadeiro_teste = f_true(x_teste)

    predicoes = np.zeros((n_simulacoes, len(x_teste)))

    for i in range(n_simulacoes):
        x_tr = np.sort(np.random.uniform(0, 2, N))
        y_tr = f_true(x_tr) + np.random.normal(0, sigma_ruido, N)
        modelo = make_pipeline(PolynomialFeatures(grau_poly), Ridge(alpha=1e-6))
        modelo.fit(x_tr.reshape(-1, 1), y_tr)
        predicoes[i] = modelo.predict(x_teste.reshape(-1, 1))

    media_pred = predicoes.mean(axis=0)
    bias2 = np.mean((f_verdadeiro_teste - media_pred) ** 2)
    variancia = np.mean(predicoes.var(axis=0))
    irred = sigma_ruido ** 2
    total = bias2 + variancia + irred
    return bias2, variancia, irred, total, predicoes, x_teste, f_verdadeiro_teste, media_pred


graus = [1, 3, 7, 14]
resultados = {g: simular_decomposicao(g) for g in graus}

# Tabela de resultados
print(f"{'Grau':<8} {'Bias²':<12} {'Variance':<12} {'Irred.':<12} {'Total':<12}")
print("-" * 56)
for g in graus:
    b2, v, ir, tot = resultados[g][:4]
    print(f"{g:<8} {b2:<12.4f} {v:<12.4f} {ir:<12.4f} {tot:<12.4f}")

In [ ]:
# Visualizando a variabilidade das previsões (Variance) em graus diferentes
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, grau in zip(axes, graus):
    b2, v, ir, tot, preds, x_t, f_t, media = resultados[grau]
    for i in range(0, 200, 10):
        ax.plot(x_t, preds[i], alpha=0.12, color='tab:blue', lw=1)
    ax.plot(x_t, media, 'tab:orange', lw=2.5, label='Média das previsões')
    ax.plot(x_t, f_t, 'k--', lw=2, label='$f$ verdadeira')
    ax.set_title(f'Grau {grau}\nBias²={b2:.3f} | Var={v:.3f}\nTotal={tot:.3f}')
    ax.set_ylim(-3, 3)
    ax.set_xlabel('x')
    if grau == graus[0]:
        ax.legend(fontsize=8)

plt.suptitle('Variabilidade das previsões entre datasets: quanto maior o grau, maior a Variance', 
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 3. Visualizando o Bias-Variance Trade-Off

### Conceito Teórico

A curva clássica do trade-off mostra como Bias² e Variance se movem em direções opostas conforme aumentamos a **complexidade do modelo** (número de parâmetros, grau do polinômio, profundidade da árvore, etc.).

**A zona ideal** fica onde a soma Bias² + Variance é mínima — nem muito simples, nem muito complexo.

### Why & How

Como o cientista de dados usa isso na prática?

1. **Diagnóstico:** compara erros de treino e validação
   - Erro treino alto + erro validação alto → **High Bias** → aumentar complexidade
   - Erro treino baixo + erro validação alto → **High Variance** → regularizar, mais dados

2. **Intervenção:** aplica a técnica certa para a causa certa

In [ ]:
# Curva clássica Bias-Variance Trade-Off
graus_todos = list(range(1, 16))
bias2_vals, var_vals, total_vals = [], [], []

for g in graus_todos:
    b2, v, ir, tot = simular_decomposicao(g)[:4]
    bias2_vals.append(b2)
    var_vals.append(v)
    total_vals.append(tot)

irred = 0.4 ** 2

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(graus_todos, bias2_vals, 'tab:red', lw=2.5, marker='o', ms=6, label='Bias²')
ax.plot(graus_todos, var_vals, 'tab:blue', lw=2.5, marker='s', ms=6, label='Variance')
ax.plot(graus_todos, total_vals, 'tab:green', lw=3, marker='^', ms=7, label='Total Error (Bias² + Var + Irred.)')
ax.axhline(irred, color='gray', ls='--', lw=1.8, label=f'Irreducible Error ($\\sigma^2={irred:.2f}$)')

idx_min = np.argmin(total_vals)
ax.axvline(graus_todos[idx_min], color='tab:green', ls=':', lw=2, alpha=0.7)
ax.annotate(f'Complexidade\nótima: grau {graus_todos[idx_min]}',
            xy=(graus_todos[idx_min], total_vals[idx_min]),
            xytext=(graus_todos[idx_min] + 1.5, total_vals[idx_min] + 0.05),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=10)

ax.annotate('High Bias\n(Underfitting)', xy=(1.5, 0.35), color='tab:red', fontsize=11, fontweight='bold')
ax.annotate('High Variance\n(Overfitting)', xy=(11, 0.35), color='tab:blue', fontsize=11, fontweight='bold')

ax.set_xlabel('Complexidade do Modelo (Grau Polinomial)', fontsize=12)
ax.set_ylabel('Erro', fontsize=12)
ax.set_title('Bias-Variance Trade-Off — A Curva Clássica', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Curvas de aprendizado: diagnóstico de Bias vs. Variance
f_true_diag = lambda x: np.sin(1.5 * np.pi * x)
X_full = np.sort(np.random.uniform(0, 2, 200)).reshape(-1, 1)
y_full = f_true_diag(X_full.ravel()) + np.random.normal(0, 0.4, 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
tamanhos = np.arange(10, 201, 10)

for ax, grau, titulo, diagnostico, cor_diag in [
    (axes[0], 1,  'Grau 1 — High Bias (Underfitting)',  'Erro treino ≈ Erro val → ambos altos', 'tab:red'),
    (axes[1], 9,  'Grau 9 — High Variance (Overfitting)', 'Gap grande entre treino e validação', 'tab:blue'),
]:
    erros_treino, erros_val = [], []
    for N in tamanhos:
        idx = np.random.choice(200, N, replace=False)
        X_tr, y_tr = X_full[idx], y_full[idx]
        modelo = make_pipeline(PolynomialFeatures(grau), Ridge(alpha=1e-4))
        cv = KFold(n_splits=5, shuffle=True, random_state=7)
        scores = cross_val_score(modelo, X_tr, y_tr, cv=cv, scoring='neg_mean_squared_error')
        modelo.fit(X_tr, y_tr)
        erros_treino.append(mean_squared_error(y_tr, modelo.predict(X_tr)))
        erros_val.append(-scores.mean())

    ax.plot(tamanhos, erros_treino, 'tab:green', lw=2, label='Erro Treino')
    ax.plot(tamanhos, erros_val, cor_diag, lw=2, label='Erro Validação (CV)')
    ax.fill_between(tamanhos, erros_treino, erros_val, alpha=0.15, color=cor_diag)
    ax.set_title(f'{titulo}\n→ {diagnostico}')
    ax.set_xlabel('Tamanho do Dataset de Treino (N)')
    ax.set_ylabel('MSE')
    ax.legend(fontsize=10)
    ax.set_ylim(0, 0.6)

plt.suptitle('Learning Curves — Diagnóstico de Bias vs. Variance', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 4. Métodos para Balancear Bias e Variance

Os slides da aula listam quatro ferramentas principais. Veja a relação de cada uma com o trade-off:

| Método | Atua sobre | Como ajuda |
|--------|-----------|------------|
| **Cross-Validation** | Avaliação | Estima o erro de generalização sem vazar o teste |
| **Regularization** | Variance | Penaliza coeficientes grandes, reduzindo overfitting |
| **Feature Selection** | Bias + Variance | Remove features irrelevantes/ruidosas |
| **Ensemble Methods** | Variance (principalmente) | Combina modelos para reduzir instabilidade |

### 4a. Cross-Validation

**Conceito:** em vez de avaliar o modelo em um único split treino/teste (que pode ser sortudo ou azarado), dividimos os dados em $K$ folds e rotacionamos quem serve de teste. A estimativa final é a média dos $K$ erros.

**Matemática:** com $K$ folds, o erro de CV é:
$$\text{CV}_{(K)} = \frac{1}{K} \sum_{k=1}^{K} \text{MSE}_k$$

**Why & How:** Cross-Validation é a **régua de medição confiável** do cientista de dados. Sem ela, você não sabe se o bom desempenho no teste foi sorte ou mérito do modelo.

In [ ]:
# Cross-Validation para selecionar o grau polinomial ideal
X_cv = np.sort(np.random.uniform(0, 2, 80)).reshape(-1, 1)
y_cv = f_true_diag(X_cv.ravel()) + np.random.normal(0, 0.4, 80)

graus_cv = range(1, 15)
erros_cv_medio = []
erros_cv_std = []

kf = KFold(n_splits=10, shuffle=True, random_state=42)

for g in graus_cv:
    modelo_cv = make_pipeline(PolynomialFeatures(g), Ridge(alpha=1e-4))
    scores = cross_val_score(modelo_cv, X_cv, y_cv, cv=kf, scoring='neg_mean_squared_error')
    erros_cv_medio.append(-scores.mean())
    erros_cv_std.append(scores.std())

erros_cv_medio = np.array(erros_cv_medio)
erros_cv_std = np.array(erros_cv_std)
melhor_grau = list(graus_cv)[np.argmin(erros_cv_medio)]

plt.figure(figsize=(10, 5))
plt.plot(graus_cv, erros_cv_medio, 'tab:blue', lw=2.5, marker='o', ms=7, label='MSE médio (10-fold CV)')
plt.fill_between(graus_cv,
                 erros_cv_medio - erros_cv_std,
                 erros_cv_medio + erros_cv_std,
                 alpha=0.2, color='tab:blue', label='±1 desvio padrão')
plt.axvline(melhor_grau, color='tab:green', ls='--', lw=2, label=f'Melhor grau: {melhor_grau}')
plt.xlabel('Grau Polinomial')
plt.ylabel('MSE (Cross-Validation)')
plt.title(f'10-Fold Cross-Validation para Seleção de Complexidade\nMelhor grau = {melhor_grau}')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()
print(f"Grau selecionado por CV: {melhor_grau} (MSE ≈ {erros_cv_medio[melhor_grau-1]:.4f})")

### 4b. Regularization (Ridge L2 e Lasso L1)

**Conceito:** adicionamos uma penalidade aos coeficientes grandes na função de custo. Isso força o modelo a ser mais "simples", reduzindo a Variance ao custo de um leve aumento no Bias.

**Matemática:**

- **Ridge (L2):** $\hat{a} = \arg\min_a \|y - Ba\|^2 + \lambda\|a\|^2$  
  Solução analítica: $\hat{a} = (B^TB + \lambda I)^{-1}B^Ty$

- **Lasso (L1):** $\hat{a} = \arg\min_a \|y - Ba\|^2 + \lambda\|a\|_1$  
  Não tem solução fechada; produz coeficientes **exatamente zero** (Feature Selection implícita)

**Why & How:** $\lambda$ é o hiperparâmetro de regularização. Maior $\lambda$ → mais restrição → menos Variance, mais Bias. Escolha via Cross-Validation em escala logarítmica.

In [ ]:
# Efeito da regularização Ridge em um modelo de alta complexidade
grau_alto = 10
lambdas = [0, 0.001, 0.01, 0.1, 1.0, 10.0]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for ax, lam in zip(axes, lambdas):
    modelo_r = make_pipeline(PolynomialFeatures(grau_alto), Ridge(alpha=lam))
    modelo_r.fit(X_cv, y_cv)
    y_pred_r = modelo_r.predict(np.linspace(0, 2, 400).reshape(-1, 1))

    cv_s = cross_val_score(modelo_r, X_cv, y_cv, cv=5, scoring='neg_mean_squared_error')
    mse_tr = mean_squared_error(y_cv, modelo_r.predict(X_cv))

    ax.scatter(X_cv.ravel(), y_cv, s=25, color='gray', alpha=0.7, zorder=5)
    ax.plot(np.linspace(0, 2, 400), f_true_diag(np.linspace(0, 2, 400)), 'k--', alpha=0.4, lw=1.5)
    ax.plot(np.linspace(0, 2, 400), y_pred_r, 'tab:blue', lw=2.5)
    ax.set_title(f'$\\lambda = {lam}$\nMSE treino={mse_tr:.3f} | CV={-cv_s.mean():.3f}')
    ax.set_ylim(-3, 3)
    ax.set_xlabel('x')

plt.suptitle(f'Ridge Regularization (L2) — Grau {grau_alto}\n$\\lambda=0$: overfitting | $\\lambda$ grande: underfitting', 
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 5. Model Selection Rules of Thumb

### Conceito Teórico

Antes de qualquer experimento, podemos usar o tamanho dos dados ($N$ = número de amostras, $p$ = número de features) como bússola inicial. Este mapa não é uma lei universal — é um **ponto de partida racional** para evitar gastar dias treinando uma rede neural quando uma regressão linear teria bastado.

### Matemática — O Risco de Dimensionalidade

A razão $N/p$ é um indicador crítico:
- Quando $N < p$ (mais features do que amostras): o sistema de equações é **subdeterminado** — existem infinitas soluções, e o risco de Variance explosiva é enorme.
- Regra prática: precisamos de pelo menos $N > 10p$ para modelos lineares sem regularização.

---

### 5a. Small Data ($N < 1000$ ou $N < p$)

**Foco principal:** manter a **Variance baixa** e evitar Overfitting.

**Por quê?** Com poucos dados, um modelo complexo vai memorizar os exemplos disponíveis e falhar completamente em dados novos. Cada dado conta — a variabilidade de qualquer subset pequeno é alta.

**Modelos recomendados:**
- **Regularized Linear Regression** (Ridge, Lasso) — baseline robusto
- **Support Vector Machines (SVM)** — bom margin em espaços de alta dimensão
- **Decision Trees pequenas** (profundidade ≤ 3-5) — interpretáveis e baixa Variance

**Heurísticas:**
- Prefira modelos simples e regularizados em vez de flexíveis
- Use Cross-Validation agressiva (5-10 folds)
- **Feature Engineering** pode trazer mais ganho do que aumentar a complexidade do modelo

### 5b. Medium Data ($1.000 < N < 100.000$)

**Foco principal:** capturar **efeitos não-lineares** mantendo interpretabilidade razoável.

**Modelos recomendados:**
- **Gradient Boosted Trees (GBT)** — especialmente para **dados tabulares**; quase sempre o melhor ponto de partida
- **Random Forest** — robusto, menos sensível a hiperparâmetros que GBT
- **Regularized Linear Models** — como baseline de comparação obrigatório

**Heurísticas:**
- Para dados tabulares: sempre tente Gradient Boosted Trees primeiro
- Se interpretabilidade for crítica: regressão regularizada + Feature Selection
- Se $p$ for grande e correlacionado: Feature Selection ou regularização L1

### 5c. Large Data ($N > 100.000$)

**Foco principal:** a **Variance se torna menos crítica** com muitos dados; a flexibilidade passa a valer o custo.

**Modelos recomendados:**
- **Neural Networks (Deep Learning)** — extraem representações hierárquicas complexas
- **Transfer Learning** — para texto e imagem, use modelos pré-treinados (Transformers, CNNs)
- **Stochastic Gradient Descent (SGD)** — para modelos lineares em escala massiva

**Heurísticas:**
- Para texto/imagem: comece sempre com modelos pré-treinados + fine-tuning
- Para dados tabulares: redes neurais só se houver razão arquitetural clara

In [ ]:
# Mapa visual das Rules of Thumb: N vs. p
fig, ax = plt.subplots(figsize=(12, 7))

# Regiões de tamanho de dados
ax.axvspan(0, 1000, alpha=0.12, color='tab:red', label='Small Data (N < 1.000)')
ax.axvspan(1000, 100000, alpha=0.12, color='tab:orange', label='Medium Data (1k < N < 100k)')
ax.axvspan(100000, 1100000, alpha=0.12, color='tab:green', label='Large Data (N > 100k)')

# Linha N = p (fronteira crítica)
p_line = np.linspace(1, 1000000, 500)
ax.plot(p_line, p_line, 'k--', lw=2, label='Linha N = p (perigo: subdeterminado)')

# Recomendações de modelos
cenarios = [
    (300,  50,  'Ridge / Lasso\nSVM\nPequenas Decision Trees', 'tab:red'),
    (300,  500, 'Lasso + Feature\nSelection\n(N < p: cuidado!)', 'darkred'),
    (10000, 20, 'Gradient Boosted Trees\nRandom Forest', 'tab:orange'),
    (10000, 5000, 'GBT + Feature\nSelection/Regulariz.', 'darkorange'),
    (500000, 50, 'Deep Learning\nTransfer Learning\nSGD linear', 'tab:green'),
]

for N, p, txt, cor in cenarios:
    ax.scatter([N], [p], s=200, color=cor, zorder=6)
    ax.annotate(txt, xy=(N, p), xytext=(N * 1.15, p * 1.3),
                fontsize=8.5, color=cor, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=cor, lw=1))

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('N (número de amostras)', fontsize=12)
ax.set_ylabel('p (número de features)', fontsize=12)
ax.set_title('Model Selection Rules of Thumb — Mapa N vs. p', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.set_xlim(50, 1100000)
ax.set_ylim(1, 10000)
plt.tight_layout()
plt.show()

In [ ]:
# Demonstração prática: comparando modelos em diferentes tamanhos de dados
def comparar_modelos_por_N(tamanhos_N, n_repeticoes=15):
    """Para cada N, treina Ridge, Random Forest e GBT e retorna MSE médio de CV."""
    modelos = {
        'Ridge (L2)':        Ridge(alpha=1.0),
        'Random Forest':     RandomForestRegressor(n_estimators=50, random_state=7),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=50, random_state=7),
    }
    resultados_N = {nome: [] for nome in modelos}

    X_pool, y_pool = make_regression(n_samples=2000, n_features=10, noise=15.0,
                                      n_informative=5, random_state=42)

    for N in tamanhos_N:
        for nome, modelo in modelos.items():
            scores_rep = []
            for _ in range(n_repeticoes):
                idx = np.random.choice(len(X_pool), N, replace=False)
                X_s, y_s = X_pool[idx], y_pool[idx]
                cv_s = cross_val_score(modelo, X_s, y_s, cv=5,
                                       scoring='neg_mean_squared_error')
                scores_rep.append(-cv_s.mean())
            resultados_N[nome].append(np.mean(scores_rep))

    return resultados_N


tamanhos = [50, 100, 250, 500, 1000, 2000]
print("Comparando modelos em diferentes tamanhos de N (aguarde ~15s)...")
res = comparar_modelos_por_N(tamanhos)

plt.figure(figsize=(10, 5))
cores_m = {'Ridge (L2)': 'tab:red', 'Random Forest': 'tab:blue', 'Gradient Boosting': 'tab:green'}
for nome, vals in res.items():
    plt.plot(tamanhos, vals, marker='o', lw=2.5, ms=8, label=nome, color=cores_m[nome])

plt.axvline(1000, color='gray', ls='--', alpha=0.7, label='Fronteira ~1k amostras')
plt.xlabel('N (número de amostras de treino)')
plt.ylabel('MSE (Cross-Validation)')
plt.title('Model Selection by Data Size:\nRidge vs. Random Forest vs. Gradient Boosting')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()
print("\nInterpretação:")
print("- Com poucos dados (N < 250): Ridge compete fortemente com métodos complexos")
print("- Com N > 500: GBT e RF começam a superar consistentemente o Ridge")
print("- A gap aumenta com N — modelos flexíveis aproveitam mais dados")

---

## 6. Data Characteristics & Constraints

### Conceito Teórico

Além do tamanho $N$, três características dos dados moldam profundamente a escolha do algoritmo:

---

### 6a. Data Type (Tipo dos dados)

| Data Type | Ponto de Partida Recomendado | Notas |
|-----------|------------------------------|-------|
| **Tabular** | **Gradient Boosted Trees** (XGBoost, LightGBM) | Quase sempre o melhor para dados tabulares |
| **Texto** | **Transfer Learning** (BERT, GPT) se N pequeno; Transformer próprio se N grande | Feature engineering manual é custoso |
| **Imagem / Áudio** | **Transfer Learning** (ResNet, EfficientNet) na quase totalidade dos casos | Dados específicos demais? Considere treinar do zero |
| **Séries Temporais** | Comece simples (ARIMA, regressão com lags) → Deep Learning só com muito dado e padrões complexos | Não pule a baseline ingênua |

---

### 6b. Noise Level / Label Quality (Qualidade dos rótulos)

**Conceito:** se os próprios rótulos $y$ estão errados ou ruidosos (Ex: anotadores humanos discordantes, sensores defeituosos), aumentar a complexidade do modelo **não ajuda** — você só vai memorizar os erros.

**Ações recomendadas:**
- **Modelos robustos** a outliers (ex: Huber Loss em vez de MSE)
- **Regularização forte** para não memorizar o ruído
- **Melhorar os labels/features** — impacto maior que qualquer truque de modelagem

---

### 6c. Interpretability (Interpretabilidade)

**Conceito:** em contextos regulados (crédito bancário, diagnóstico médico, decisões jurídicas), não basta que o modelo acerte — é preciso **explicar por que** ele acertou.

| Necessidade | Modelos adequados |
|-------------|------------------|
| Alta interpretabilidade, baixa latência | Regressão Linear/Logística, Decision Trees rasas |
| Alta acurácia + interpretabilidade parcial | GBT com SHAP values |
| Máxima acurácia, caixa-preta aceitável | Deep Learning, Random Forest |

### Why & How

A mensagem central dos slides é clara: **não existe algoritmo universalmente superior**. A escolha ótima emerge da combinação de $N$, $p$, tipo de dado, nível de ruído e restrições de negócio. Um cientista de dados sênior raciocina sobre esses eixos *antes* de abrir o código.

In [ ]:
# Efeito do Noise Level no desempenho de modelos simples vs. complexos
f_noise = lambda x: 0.5 * x[:, 0] + 0.3 * x[:, 1] ** 2

niveis_ruido = [0.5, 2.0, 5.0, 10.0]
N_ruido = 300

modelos_ruido = {
    'Ridge (simples)':  Ridge(alpha=1.0),
    'GBT (complexo)':   GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42),
}

X_r = np.random.randn(N_ruido, 5)
resultados_ruido = {nome: [] for nome in modelos_ruido}

for sigma in niveis_ruido:
    y_r = f_noise(X_r) + np.random.normal(0, sigma, N_ruido)
    for nome, modelo in modelos_ruido.items():
        scores = cross_val_score(modelo, X_r, y_r, cv=5, scoring='neg_mean_squared_error')
        resultados_ruido[nome].append(-scores.mean())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for nome, vals in resultados_ruido.items():
    ax.plot(niveis_ruido, vals, marker='o', lw=2.5, ms=8, label=nome)
ax.set_xlabel('Nível de Ruído ($\\sigma$)')
ax.set_ylabel('MSE (Cross-Validation)')
ax.set_title('Noise Level: modelo complexo\naproveita menos com ruído alto')
ax.legend(fontsize=10)

# Diferença relativa: vantagem do GBT sobre Ridge
ax = axes[1]
vantagem = [resultados_ruido['Ridge (simples)'][i] - resultados_ruido['GBT (complexo)'][i]
            for i in range(len(niveis_ruido))]
bars = ax.bar(niveis_ruido, vantagem, color=['tab:green' if v > 0 else 'tab:red' for v in vantagem],
              width=0.8, alpha=0.8)
ax.axhline(0, color='black', lw=1)
ax.set_xlabel('Nível de Ruído ($\\sigma$)')
ax.set_ylabel('MSE Ridge − MSE GBT')
ax.set_title('Vantagem do GBT sobre Ridge por nível de ruído\n(verde = GBT melhor, vermelho = Ridge melhor)')
for bar, v in zip(bars, vantagem):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{v:+.2f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()
print("Insight: com ruído alto, a vantagem de modelos complexos diminui — vale a pena melhorar os labels!")

---

## 7. Structured 4-Phase Model Selection Workflow

### Conceito Teórico

O professor apresenta um **framework corporativo** de 4 fases para seleção de modelos. Este processo é o que separa o cientista de dados júnior (que testa algoritmos aleatoriamente) do sênior (que usa um método sistemático e justificado).

---

### Phase 1: Define Decision Criteria

**O que fazer:** antes de tocar nos dados, traduza os objetivos de negócio em **métricas mensuráveis** e documente restrições.

Perguntas a responder:
- Qual é a **métrica de sucesso**? (MSE? Accuracy? AUC? F1? Precision@K?)
- Existem **restrições de latência**? (O modelo precisa responder em <10ms?)
- Existem **restrições de interpretabilidade**? (Setor regulado?)
- Qual é o **custo de erro**? (Falso positivo é mais caro que falso negativo?)
- Há problemas de **Class Imbalance**? (1 positivo para 1.000 negativos?)

---

### Phase 2: Characterize Data & Problem

**O que fazer:** análise exploratória focada em informações que guiam a seleção do modelo.

- Tamanho: $N$ e $p$ (calcule a razão $N/p$)
- Tipo de dado: tabular, texto, imagem, série temporal
- Qualidade: verifique proporção de valores ausentes, outliers, ruído nos labels
- Estrutura: features correlacionadas? Relações lineares ou não-lineares?

---

### Phase 3: Establish Baselines

**O que fazer:** crie **dois modelos de referência** antes de qualquer Fine-Tuning.

- **Dummy Baseline:** o modelo mais ingênuo possível (ex: prever sempre a média para regressão). Define o piso mínimo de desempenho.
- **ML Baseline:** um modelo linear simples com padrões padrão (Ridge, Logistic Regression). Define o que é "fácil de alcançar".

> **Metacognição do Cientista Preguiçoso:** se um modelo linear atinge 85% de acurácia em 2 segundos e uma rede neural leva 3 horas para atingir 86%, o ganho marginal de 1% justifica a complexidade de manutenção e os custos de infraestrutura? **Estabelecer uma boa Baseline evita gastar energia à toa.**

---

### Phase 4: Candidate Shortlist

**O que fazer:** usando as heurísticas (Seção 5), reduza o espaço de busca para **2 a 4 famílias de modelos** antes de qualquer Fine-Tuning de hiperparâmetros.

**Por que limitar?** Fazer Fine-Tuning em 10 famílias de modelos é desperdício de tempo computacional e aumenta o risco de vazamento do conjunto de teste (overfitting de hiperparâmetros). Primeiro descarte os candidatos fracos com avaliação rápida (CV com hiperparâmetros padrão), depois refine os 2-4 finalistas.

In [ ]:
# Implementando o workflow completo nas 4 fases
print("="*65)
print("STRUCTURED 4-PHASE MODEL SELECTION WORKFLOW")
print("="*65)

# --- Dados simulados (problema de regressão médio porte) ---
X_wf, y_wf = make_regression(n_samples=2500, n_features=15, n_informative=8,
                              noise=20.0, random_state=42)

# Separar teste ANTES de qualquer análise (nunca tocar no teste até o final)
from sklearn.model_selection import train_test_split
X_dev, X_test, y_dev, y_test = train_test_split(X_wf, y_wf, test_size=0.2, random_state=42)

N, p = X_dev.shape

print("\n📋 PHASE 1 — Decision Criteria")
print("-" * 40)
print("  Problema:       Regressão (prever valor contínuo)")
print("  Métrica:        MSE (Mean Squared Error)")
print("  Restrições:     Sem requisito de latência, interpretabilidade moderada")
print("  Class Imbalance: N/A (regressão)")

print("\n📊 PHASE 2 — Characterize Data")
print("-" * 40)
print(f"  N (amostras dev):  {N}")
print(f"  p (features):      {p}")
print(f"  Razão N/p:         {N/p:.1f}  → {'Seguro para modelos flexíveis' if N/p > 10 else 'Cuidado — pode precisar regularização'}")
print(f"  Faixa de dados:    {'Medium Data (1k-100k)' if 1000 < N < 100000 else 'Outro'}")
print(f"  Tipo de dado:      Tabular")
print(f"  Recomendação:      Gradient Boosted Trees, Random Forest")

In [ ]:
print("\n🎯 PHASE 3 — Establish Baselines")
print("-" * 40)

cv_wf = KFold(n_splits=5, shuffle=True, random_state=42)

# Dummy Baseline: sempre prevê a média
dummy = DummyRegressor(strategy='mean')
scores_dummy = cross_val_score(dummy, X_dev, y_dev, cv=cv_wf, scoring='neg_mean_squared_error')
mse_dummy = -scores_dummy.mean()

# ML Baseline: Ridge simples
ridge_base = Ridge(alpha=1.0)
scores_ridge = cross_val_score(ridge_base, X_dev, y_dev, cv=cv_wf, scoring='neg_mean_squared_error')
mse_ridge = -scores_ridge.mean()

print(f"  Dummy Baseline (média):   MSE = {mse_dummy:.2f}")
print(f"  ML Baseline (Ridge L2):   MSE = {mse_ridge:.2f}")
print(f"  Melhoria do Ridge vs. Dummy: {(1 - mse_ridge/mse_dummy)*100:.1f}% de redução do MSE")
print()
print("  → O Ridge reduz significativamente o erro. Qualquer candidato")
print("    deve superar o Ridge para justificar maior complexidade.")

In [ ]:
print("\n🔍 PHASE 4 — Candidate Shortlist (2-4 famílias, hiperparâmetros padrão)")
print("-" * 58)

candidatos = {
    'Ridge (Baseline ML)':      Ridge(alpha=1.0),
    'Lasso (L1)':               Lasso(alpha=0.1, max_iter=5000),
    'Random Forest':            RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':        GradientBoostingRegressor(n_estimators=100, random_state=42),
}

resultados_cand = {}
for nome, modelo in candidatos.items():
    s = cross_val_score(modelo, X_dev, y_dev, cv=cv_wf, scoring='neg_mean_squared_error')
    resultados_cand[nome] = {'mse': -s.mean(), 'std': s.std()}

print(f"  {'Modelo':<28} {'MSE (CV)':<14} {'±Std':<10} {'vs. Dummy':<15} {'vs. Ridge ML'}")
print("  " + "-"*80)
for nome, res in sorted(resultados_cand.items(), key=lambda x: x[1]['mse']):
    ganho_dummy = (1 - res['mse']/mse_dummy)*100
    ganho_ridge = (1 - res['mse']/mse_ridge)*100
    status = "★ CANDIDATO" if res['mse'] < mse_ridge * 0.98 else "  baseline"
    print(f"  {nome:<28} {res['mse']:<14.2f} {res['std']:<10.2f} "
          f"{ganho_dummy:<15.1f}% {ganho_ridge:.1f}%  {status}")

melhor = min(resultados_cand, key=lambda x: resultados_cand[x]['mse'])
print(f"\n  Finalistas para Fine-Tuning: Random Forest e Gradient Boosting")
print(f"  → Melhor candidato padrão: {melhor} (MSE = {resultados_cand[melhor]['mse']:.2f})")

In [ ]:
# Avaliação final no conjunto de teste (UMA ÚNICA VEZ)
print("\n🏆 AVALIAÇÃO FINAL NO CONJUNTO DE TESTE")
print("(Executar apenas uma vez — após decidir o modelo!)")
print("-" * 50)

modelo_final = GradientBoostingRegressor(n_estimators=100, random_state=42)
modelo_final.fit(X_dev, y_dev)
mse_teste = mean_squared_error(y_test, modelo_final.predict(X_test))
mse_dummy_teste = mean_squared_error(y_test, np.full_like(y_test, y_dev.mean()))

print(f"  Dummy Baseline (teste):     MSE = {mse_dummy_teste:.2f}")
print(f"  GBT Final (teste):          MSE = {mse_teste:.2f}")
print(f"  Melhoria final: {(1 - mse_teste/mse_dummy_teste)*100:.1f}% redução no MSE")
print()
print("  ⚠️  O teste só foi aberto UMA VEZ. Este é o resultado oficial.")

In [ ]:
# Visualização comparativa do workflow completo
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gráfico 1: comparação dos candidatos
ax = axes[0]
nomes = list(resultados_cand.keys())
mses  = [resultados_cand[n]['mse'] for n in nomes]
stds  = [resultados_cand[n]['std'] for n in nomes]
cores_bar = ['tab:gray' if m >= mse_ridge * 0.99 else 'tab:blue' for m in mses]
cores_bar[nomes.index('Gradient Boosting')] = 'tab:green'

bars = ax.barh(nomes, mses, xerr=stds, color=cores_bar, alpha=0.85, capsize=5)
ax.axvline(mse_dummy, color='red', ls='--', lw=2, label=f'Dummy Baseline ({mse_dummy:.1f})')
ax.axvline(mse_ridge, color='orange', ls='--', lw=2, label=f'Ridge Baseline ({mse_ridge:.1f})')
ax.set_xlabel('MSE (5-fold Cross-Validation)')
ax.set_title('Phase 4: Candidate Shortlist\n(verde = finalista selecionado)')
ax.legend(fontsize=9)

# Gráfico 2: Resumo das 4 fases como fluxograma textual
ax = axes[1]
ax.axis('off')
fases = [
    ('Phase 1', 'Define Decision Criteria',
     'Métrica: MSE\nRestrições: interpretabilidade moderada\nClass Imbalance: N/A'),
    ('Phase 2', 'Characterize Data & Problem',
     f'N={N}, p={p}, N/p={N/p:.0f}\nTipo: Tabular → GBT recomendado\nDado: Medium Data'),
    ('Phase 3', 'Establish Baselines',
     f'Dummy: MSE={mse_dummy:.1f}\nRidge ML: MSE={mse_ridge:.1f}\nMelhoria baseline: {(1-mse_ridge/mse_dummy)*100:.0f}%'),
    ('Phase 4', 'Candidate Shortlist (2-4)',
     f'Finalistas: RF + GBT\nMelhor (CV): {melhor}\nMSE teste final: {mse_teste:.1f}'),
]

cores_fase = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
for i, (fase, titulo, detalhe) in enumerate(fases):
    y_pos = 0.85 - i * 0.23
    rect = mpatches.FancyBboxPatch((0.02, y_pos - 0.09), 0.96, 0.18,
                                    boxstyle='round,pad=0.02',
                                    facecolor=cores_fase[i], alpha=0.15,
                                    edgecolor=cores_fase[i], lw=2,
                                    transform=ax.transAxes)
    ax.add_patch(rect)
    ax.text(0.05, y_pos + 0.05, f'{fase}: {titulo}',
            transform=ax.transAxes, fontsize=10, fontweight='bold',
            color=cores_fase[i], va='top')
    ax.text(0.05, y_pos - 0.02, detalhe,
            transform=ax.transAxes, fontsize=8.5, va='top', color='#333333')
    if i < 3:
        ax.annotate('', xy=(0.5, y_pos - 0.085), xytext=(0.5, y_pos - 0.095),
                    xycoords='axes fraction', textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='gray', lw=2))

ax.set_title('4-Phase Workflow — Resumo', fontsize=11, fontweight='bold')

plt.suptitle('Model Selection Workflow Completo', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 8. Metacognição: A Mentalidade do Cientista Preguiçoso

> *"Se um modelo linear simples (ML Baseline) atinge 85% de precisão em 2 segundos de computação, e uma rede neural complexa leva 3 horas para atingir 86%, o ganho marginal de 1% justifica a complexidade de manutenção e custos de infraestrutura?"*

Esta é a pergunta que todo cientista de dados deveria se fazer **antes** de avançar para modelos mais complexos.

### O Princípio da Navalha de Occam Aplicado ao ML

Prefira o modelo mais simples que satisfaça os critérios de decisão definidos na **Phase 1**. Complexidade extra tem custos reais:

| Custo da complexidade | Impacto |
|-----------------------|---------|
| Tempo de treinamento | Horas → Dias para grandes redes |
| Custo computacional | GPU/TPU são caros |
| Manutenção | Modelos complexos quebram de formas mais sutis |
| Interpretabilidade | Difícil de auditar e debugar |
| Estabilidade | Alta Variance: pequenas mudanças nos dados → grandes mudanças no modelo |

### O Fluxo Correto de Decisão

In [ ]:
# Visualizando o trade-off entre acurácia e custo computacional
import time

X_meta, y_meta = make_regression(n_samples=5000, n_features=20, noise=10.0,
                                  n_informative=8, random_state=42)
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(X_meta, y_meta, test_size=0.2, random_state=42)

experimentos = [
    ('Dummy (média)',          DummyRegressor(strategy='mean')),
    ('Ridge (simples)',        Ridge(alpha=1.0)),
    ('Random Forest (50 árv)', RandomForestRegressor(n_estimators=50, random_state=42)),
    ('GBT (100 árv, depth=4)', GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42)),
    ('GBT (500 árv, depth=6)', GradientBoostingRegressor(n_estimators=500, max_depth=6, random_state=42)),
]

dados_exp = []
for nome, modelo in experimentos:
    t0 = time.time()
    modelo.fit(X_tr_m, y_tr_m)
    tempo_treino = time.time() - t0
    mse_m = mean_squared_error(y_te_m, modelo.predict(X_te_m))
    dados_exp.append({'nome': nome, 'mse': mse_m, 'tempo': tempo_treino})

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

nomes_e = [d['nome'] for d in dados_exp]
mses_e  = [d['mse'] for d in dados_exp]
tempos_e = [d['tempo'] for d in dados_exp]

# Gráfico 1: MSE
ax = axes[0]
cores_e = ['tab:gray', 'tab:orange', 'tab:blue', 'tab:green', 'tab:purple']
bars_mse = ax.barh(nomes_e, mses_e, color=cores_e, alpha=0.85)
for bar, v in zip(bars_mse, mses_e):
    ax.text(bar.get_width() + max(mses_e)*0.01, bar.get_y() + bar.get_height()/2,
            f'{v:.1f}', va='center', fontsize=10)
ax.set_xlabel('MSE (Teste)')
ax.set_title('Desempenho (MSE no Teste)')
ax.invert_xaxis()

# Gráfico 2: Tempo de treino
ax = axes[1]
bars_t = ax.barh(nomes_e, tempos_e, color=cores_e, alpha=0.85)
for bar, v in zip(bars_t, tempos_e):
    ax.text(bar.get_width() + max(tempos_e)*0.01, bar.get_y() + bar.get_height()/2,
            f'{v:.2f}s', va='center', fontsize=10)
ax.set_xlabel('Tempo de Treino (segundos)')
ax.set_title('Custo Computacional (Tempo de Treino)')

plt.suptitle('Navalha de Occam no ML: Ganho Marginal vs. Custo Marginal\n'
             'A Mentalidade do Cientista Preguiçoso', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nReflexão:")
ganho_ridge_vs_dummy = (mses_e[0] - mses_e[1]) / mses_e[0] * 100
ganho_gbt_vs_ridge   = (mses_e[1] - mses_e[3]) / mses_e[1] * 100
custo_extra          = tempos_e[3] / max(tempos_e[1], 0.001)
print(f"  Ridge vs. Dummy:          -{ganho_ridge_vs_dummy:.0f}% MSE em {tempos_e[1]:.3f}s")
print(f"  GBT vs. Ridge:            -{ganho_gbt_vs_ridge:.0f}% MSE em {custo_extra:.0f}x mais tempo")
print(f"  GBT 500 vs. GBT 100:      {(mses_e[3]-mses_e[4])/mses_e[3]*100:.1f}% melhoria com {tempos_e[4]/max(tempos_e[3],0.001):.0f}x mais tempo")
print()
print("  → Pergunte sempre: 'O ganho marginal justifica o custo marginal?'")

---

## Resumo Final — O que aprendemos

| Conceito | Ideia Central | Fórmula/Ferramenta |
|----------|---------------|--------------------|
| **Core Dilemma** | Data Fit vs. Generalization são conflitantes | Curvas de aprendizado |
| **Error Decomposition** | Todo erro tem 3 causas distintas | $\text{MSE} = \text{Bias}^2 + \text{Variance} + \sigma^2$ |
| **Bias** | Modelo sistematicamente errado | $(f - \mathbb{E}[\hat{f}])^2$ |
| **Variance** | Modelo sensível ao dataset de treino | $\mathbb{E}[(\hat{f} - \mathbb{E}[\hat{f}])^2]$ |
| **Irreducible Error** | Ruído intrínseco — nada a fazer | $\sigma^2$ |
| **Cross-Validation** | Estimador confiável do erro real | $\text{CV}_K = \frac{1}{K}\sum_k \text{MSE}_k$ |
| **Regularization** | Reduz Variance penalizando coeficientes | Ridge ($\lambda\|a\|^2$), Lasso ($\lambda\|a\|_1$) |
| **Small Data** | Priorize Variance baixa | Ridge, SVM, Decision Trees rasas |
| **Medium Data** | Efeitos não-lineares + interpretabilidade | Gradient Boosted Trees, Random Forest |
| **Large Data** | Flexibilidade vale o custo | Deep Learning, Transfer Learning, SGD |
| **4-Phase Workflow** | Framework sistemático corporativo | Define → Characterize → Baseline → Shortlist |

---

## Exercícios para Fixar

1. **Decomposição empírica:** na Seção 2, aumente `sigma_ruido` de 0.4 para 1.0. Como o Irreducible Error e o Total Error mudam? O Bias² e a Variance mudam também?

2. **Diagnóstico de Bias/Variance:** gere um dataset com $N=50$ e ajuste um Decision Tree com `max_depth=None` (profundidade ilimitada). Plote a Learning Curve. Qual é o diagnóstico?

3. **Escolha de $\lambda$:** na Seção 4b, use `cross_val_score` para encontrar o melhor $\lambda$ para Ridge em uma grade logarítmica de `[0.001, 0.01, 0.1, 1, 10, 100]`. Qual $\lambda$ minimiza o MSE de CV?

4. **Workflow completo:** aplique o 4-Phase Workflow ao dataset `load_diabetes()` do sklearn. Documente cada fase antes de escrever código.

5. **N vs. p (Overfitting garantido):** crie um dataset com $N=30$ e $p=50$ (mais features que amostras). Tente ajustar uma regressão linear sem regularização. O que acontece? Agora use Lasso — qual $\alpha$ funciona melhor?

---

### Referências

- `L1_General_Linear_Models_Basis_Functions.ipynb` — fundamentos de regressão e Basis Functions
- `visao_probabilistica.ipynb` — perspectiva probabilística (MAP, regularização como prior)
- `lectures/Statistisches_Lernen_2_v2.pdf` — slides originais desta aula
- Hastie, Tibshirani & Friedman: *The Elements of Statistical Learning* — Capítulo 7 (Model Assessment)
- James et al.: *An Introduction to Statistical Learning* — Capítulo 2 (Statistical Learning) e Capítulo 5 (Resampling)